# Module 7.2: Advanced Attention (MQA & GQA)

Welcome to the second notebook of Module 7!

In the previous notebook, we cut out the repeated $O(N^2)$ recalculation using **KV Caching**. However, we traded *compute time* for *memory space* (RAM/VRAM).

In this notebook, we look at how modern models modify the Attention mechanism to keep the KV cache small using **Multi-Query Attention (MQA)** and **Grouped-Query Attention (GQA)**.

## 1. The Memory Bottleneck of Multi-Head Attention (MHA)

In standard Multi-Head Attention (which we built in Module 3.1), each Attention "Head" gets its own dedicated: 
- `Q` (Query) matrix
- `K` (Key) matrix
- `V` (Value) matrix

When doing KV-Caching, this means **every single head** is saving its own `K` and `V` vectors to memory for every single token generated. With 32 Heads (like Llama 2 7B), you are duplicating the cached Keys and Values across all 32 heads.

```mermaid
graph LR
    A[Head 1] -->|Caches| K1[Key 1]
    A -->|Caches| V1[Value 1]
    B[Head 2] -->|Caches| K2[Key 2]
    B -->|Caches| V2[Value 2]
    C[Head N] -->|Caches| KN[Key N]
    C -->|Caches| VN[Value N]
```

## 2. Multi-Query Attention (MQA)

**MQA** proposes a radical solution to the memory bloat: What if *every* Attention Head still gets its own Query (Q), but they all **share a single Key (K) and a single Value (V)** projection?

```mermaid
graph LR
    A[Head 1 Query] -->|Shares| K[Shared Key Projection]
    B[Head 2 Query] -->|Shares| K
    C[Head N Query] -->|Shares| K
    
    A -->|Shares| V[Shared Value Projection]
    B -->|Shares| V
    C -->|Shares| V
```

### Why do this?
If we only have 1 `K` vector and 1 `V` vector instead of 32, our KV Cache becomes about **32 times smaller**! That frees up a lot of VRAM, which helps us fit longer contexts and serve more users at once. (A smaller cache helps with long context, but on its own it isn't the only thing that determines the maximum context length.)

*Note: This usually hurts model quality a little, because the heads can no longer express diverse Key/Value concepts — they all read from the same shared memory. But the memory savings are large.*

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_model = d_model
        
        # The magic of MQA:
        # Query still projects to the full d_model (each head gets its own slice)
        self.W_q = nn.Linear(d_model, d_model)
        
        # Key and Value ONLY project to the size of a SINGLE head! (Shared!)
        self.W_k = nn.Linear(d_model, self.d_head)
        self.W_v = nn.Linear(d_model, self.d_head)
        
        # Output projection: mixes the concatenated heads back together (same as MHA in Module 5)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, seq_len, d_model = x.shape
        
        # Shape Q: (Batch, Num_Heads, Seq_Len, Head_Dim)
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        
        # Shape K, V: (Batch, 1, Seq_Len, Head_Dim) -> Notice the "1"!
        # We simulate the "sharing" by creating a pseudo num_heads dimension of size 1
        K = self.W_k(x).view(batch, seq_len, 1, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_len, 1, self.d_head).transpose(1, 2)
        
        # PyTorch will automatically stretch (broadcast) the "1" dimension of K and V 
        # to match the "num_heads" dimension of Q during the dot product!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)
        # Proof the broadcast worked: scores has a slot for EVERY query head,
        # even though we only stored ONE shared K. Shape: (Batch, Num_Heads, Seq, Seq)
        print(f"scores.shape: {scores.shape}  (note: {self.num_heads} head slots from a single shared K)")
        weights = torch.softmax(scores, dim=-1)
        
        # Per-head output: (Batch, Num_Heads, Seq_Len, Head_Dim)
        context = torch.matmul(weights, V)
        
        # Merge the heads back: transpose then reshape to (Batch, Seq_Len, d_model),
        # then apply the output projection W_o.
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        output = self.W_o(context)
        return output
        
mqa_layer = MultiQueryAttention(d_model=512, num_heads=8)
dummy_input = torch.randn(2, 10, 512)
out = mqa_layer(dummy_input)
print(f"MQA Output shape: {out.shape} -> back to (Batch, Seq, d_model), ready for the next layer!")

## 3. Grouped-Query Attention (GQA)

If normal Attention (MHA) has the best quality but the largest cache, and MQA has the smallest cache but slightly worse quality, what do we do?

**GQA** is the middle ground. Instead of 1 shared K/V, or 32 unshared K/Vs, we split the difference. If we have 32 queries, we group them into 8 groups of 4. Each group gets its own shared K and V.

- **MHA** (Full Quality): 32 Qs, 32 Ks, 32 Vs
- **MQA** (Smallest Cache): 32 Qs, 1 K, 1 V
- **GQA** (Goldilocks): 32 Qs, 8 Ks, 8 Vs. (Used by Llama 2's larger models and all of Llama 3. The smaller Llama 2 models — 7B and 13B — still use plain 32-head MHA.)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_groups=4):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.d_head = d_model // num_heads
        self.d_model = d_model
        
        # Validate division
        assert num_heads % num_kv_groups == 0, "Heads must be evenly grouped!"
        self.heads_per_group = num_heads // num_kv_groups

        self.W_q = nn.Linear(d_model, d_model)
        
        # K and V project to the size of the GROUPS, not the individual heads!
        d_kv = num_kv_groups * self.d_head
        self.W_k = nn.Linear(d_model, d_kv)
        self.W_v = nn.Linear(d_model, d_kv)
        
        # Output projection: mixes the concatenated heads back together
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, seq_len, d_model = x.shape
        
        # Shape Q: (Batch, Num_Heads, Seq_Len, Head_Dim)
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        
        # Shape K, V: (Batch, Groups, Seq_Len, Head_Dim)
        K = self.W_k(x).view(batch, seq_len, self.num_kv_groups, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_len, self.num_kv_groups, self.d_head).transpose(1, 2)
        
        # To do the dot product, we use `torch.repeat_interleave` to copy each
        # group's K and V so they line up with the Num_Heads dimension.
        # Ex: 4 groups, 8 heads total -> each group repeats twice.
        # Result: 8 K and V tensors, ready for standard Multi-Head Attention!
        K_expanded = torch.repeat_interleave(K, repeats=self.heads_per_group, dim=1)
        V_expanded = torch.repeat_interleave(V, repeats=self.heads_per_group, dim=1)
        
        # Regular Attention Math!
        scores = torch.matmul(Q, K_expanded.transpose(-2, -1)) / (self.d_head ** 0.5)
        print(f"scores.shape: {scores.shape}  ({self.num_heads} head slots from {self.num_kv_groups} KV groups)")
        weights = torch.softmax(scores, dim=-1)
        
        # Per-head output: (Batch, Num_Heads, Seq_Len, Head_Dim)
        context = torch.matmul(weights, V_expanded)
        
        # Merge the heads back to (Batch, Seq_Len, d_model) and apply W_o.
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        output = self.W_o(context)
        return output
        
# 8 Queries, grouped into 4 groups (so every 2 query heads share one K/V group)
gqa_layer = GroupedQueryAttention(d_model=512, num_heads=8, num_kv_groups=4)
dummy_input = torch.randn(2, 10, 512)
out = gqa_layer(dummy_input)
print(f"GQA Output shape: {out.shape} -> back to (Batch, Seq, d_model)!")

## Summary

You've now seen the three flavors of attention used in modern Transformers:

- **MHA** — every head has its own K/V (best quality, biggest cache).
- **MQA** — all heads share one K/V (smallest cache, slight quality cost).
- **GQA** — heads grouped to share K/V (the practical middle ground used by today's large models).

The neat part: these are all the *same* mechanism with one knob — the number of KV groups. The next notebook, **Module 7.3: FlashAttention**, goes one level deeper to the GPU hardware itself, making attention faster by rethinking how memory is read.

### 🏋️ Try it yourself

GQA, MHA, and MQA are really one design with a single dial: `num_kv_groups`.

- Set `num_kv_groups = num_heads` → every head gets its own K/V → that's plain **MHA**.
- Set `num_kv_groups = 1` → all heads share one K/V → that's **MQA**.

Run `GroupedQueryAttention` at both extremes and watch the `scores.shape` printout stay the same (the head dimension never changes) — only the *number of stored KV groups* changes.

In [ ]:
# Your turn: one class, three behaviors. Watch the scores.shape stay identical.
dummy_input = torch.randn(2, 10, 512)

print("MHA  (num_kv_groups = num_heads = 8):")
GroupedQueryAttention(d_model=512, num_heads=8, num_kv_groups=8)(dummy_input)

print("\nGQA  (num_kv_groups = 4):")
GroupedQueryAttention(d_model=512, num_heads=8, num_kv_groups=4)(dummy_input)

print("\nMQA  (num_kv_groups = 1):")
GroupedQueryAttention(d_model=512, num_heads=8, num_kv_groups=1)(dummy_input)

print("\nSame scores.shape every time — only the number of stored KV groups changed.")